# DreamOn Java-Identifier benchmark (Colab)

EM + LLM-as-Judge for the two AISE-TUDelft models on the RefineID task:
* `AISE-TUDelft/dreamon-0.5b-Java-Identifiers`  (1.16 GB, fixed-length diffusion)
* `AISE-TUDelft/dreamon-7b-Java-Identifiers`   (15.2 GB, DreamOn variable-length)

**Runtime:** `Runtime > Change runtime type` → **A100** or **L4** for the 7B (needs ~16 GB VRAM in bf16). **T4** is fine for the 0.5B only.

Both EM and LJ outputs go under **`results/dreamon_java/`** (EM CSVs, `judge_inputs/`, `llm_judge/`), so zipping the whole `results/` folder (step 7) captures everything for the Drive upload. The model under test gets **no prompt** — pure masked infilling, consistent with our previous dLLM setup.

EM metrics per sample (all `[MASK]` in a sample are the SAME variable): `first_mask` (matches our existing Table-1 rule) · `majority` · `any` · **`all_sites_consistent`** · **`all_sites_correct`** (strict EM).

The optimizer/training-state blobs in the repos (2.3 GB / 30.5 GB) are skipped automatically.

## 0. GPU check

In [ ]:
!nvidia-smi

## 1. Install dependencies
`transformers==4.46.2` matches the version the Dream custom code targets — do not upgrade it.

In [ ]:
!pip install -q "transformers==4.46.2" accelerate huggingface_hub omegaconf tqdm pandas
import transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Get the code + data
Clones the repo (brings `data/test.csv` and all experiment scripts). Re-running pulls the latest.

In [ ]:
import os
REPO = '/content/CoRefusion'
if not os.path.exists(REPO):
    # If the repo is PRIVATE, use:  https://<GITHUB_TOKEN>@github.com/ANONYMOUS/CoReFusion
    !git clone https://github.com/ANONYMOUS/CoReFusion $REPO
else:
    !cd $REPO && git pull --ff-only
%cd $REPO
!ls experiments/benchmark_dreamon_java_identifiers.py experiments/run_llm_judge_dreamon_java.py data/test.csv

## 3. (Optional) Hugging Face login
The two model repos are **public**, so download needs no token. Only run this if you want to **upload results** to a HF dataset (`--hf-repo`) or if the repos become gated.

In [ ]:
import getpass, os
from huggingface_hub import login
tok = getpass.getpass('HF token (press Enter to skip): ').strip()
if tok:
    os.environ['HF_TOKEN'] = tok
    login(tok)
    print('logged in')
else:
    print('skipped (download still works for public repos)')

## 4. Smoke test (5 samples, --debug)
Validates that each model loads and `diffusion_generate` produces sane identifiers BEFORE the full run. The `[sanity]` line should fill `return a + [MASK];` with something like `a`/`b`/`1`.

In [ ]:
!python experiments/benchmark_dreamon_java_identifiers.py --model dreamon-0.5b-Java --max-samples 5 --debug

In [ ]:
!python experiments/benchmark_dreamon_java_identifiers.py --model dreamon-7b-Java --max-samples 5 --debug

## 5. Full EM benchmark (1000 samples, both models)
Add `--hf-repo anonymous/benchmark_ReFineID_DreamOn_Java` to auto-upload (needs the token from step 3).
Tip: run one model at a time with `--model dreamon-7b-Java` if VRAM is tight.

In [ ]:
!python experiments/benchmark_dreamon_java_identifiers.py

### EM summary

In [ ]:
import glob, pandas as pd
f = sorted(glob.glob('results/dreamon_java/summary_*.csv'))[-1]
print(f)
pd.read_csv(f)[['model','gen_mode','site_acc','first_mask_acc','majority_acc','any_acc','consistent_rate','all_correct_acc','errors']]

## 6. LLM-as-Judge (Qwen2.5-7B-Instruct)
Pure-Python runner (no bash). Builds baseline CSVs (first_mask + majority) and judges each with the **same** logic as our previous LJ — same prompt, same Qwen judge, `--resume`, exact-match shortcut. The judge model is loaded once and reused. Everything lands under `results/dreamon_java/` so step 7's zip captures it.

Options: `--max-samples 50` (quick test) · `--judge-model Qwen2.5-14B-Instruct`.

In [ ]:
!python experiments/run_llm_judge_dreamon_java.py

### LLM-as-Judge summary

In [ ]:
import glob, pandas as pd
fs = sorted(glob.glob('results/dreamon_java/llm_judge/summary_*.csv'))
pd.read_csv(fs[-1]) if fs else 'no judge summary found'

## 7. Save results to Google Drive
Zips the whole `results/` folder (EM + judge) and copies it to `MyDrive`. Colab is ephemeral — do this before disconnecting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import shutil
from datetime import datetime

folder = "results"
if not os.path.exists(folder):
    print(f"\u00d7 \u6587\u4ef6\u5939 {folder} \u4e0d\u5b58\u5728")
else:
    zip_name = f"results_{datetime.now().strftime('%Y%m%d_%H%M')}_language_model_as_judge"
    shutil.make_archive(zip_name, 'zip', folder)
    !cp {zip_name}.zip /content/drive/MyDrive/
    print(f"\u221a \u5df2\u4e0a\u4f20\uff1a/MyDrive/{zip_name}.zip")